# Flat PPO: Config C Matched-Compute Baseline

**Purpose:** controlled flat/mixed PPO baseline for the curriculum analysis.

This notebook is derived from the original Config C training setup. The PPO configuration,
reward functions, model, tokenizer, and generation settings are kept unchanged.

**Only the curriculum mechanism is removed:**
- all Easy + Medium + Hard training samples are pooled from the start;
- no Phase 1 / Phase 2 / Phase 3 training;
- the reward vector is fixed to Config C's **Phase 3** weights;
- the total number of PPO updates is matched to the **actual** Phase 1 + Phase 2 + Phase 3
  updates already completed by each Config C seed.

Run on **NVIDIA T4** only.

## 1. Install, mount Drive, and verify T4

In [1]:
!pip install -q "trl==0.11.4" transformers sentence-transformers accelerate pandas scipy matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os, time, math, random, logging, json, re, glob, gc
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("Flat-PPO")

assert torch.cuda.is_available(), "CUDA GPU is required."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
assert "T4" in GPU_NAME.upper(), f"Expected NVIDIA T4, but found: {GPU_NAME}"

DEVICE = "cuda"
print("CUDA device:", torch.cuda.current_device())
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 11.3 MB/s eta 0:00:00
Mounted at /content/drive
GPU: Tesla T4
CUDA device: 0
VRAM (GB): 14.56


## 2. Paths and seeds

In [2]:
DATA_DIR = "/content/drive/MyDrive/Data_Khoaluan/03_stratified_benchmark"
T5_SFT_DIR = "/content/drive/MyDrive/Data_Khoaluan/T5_base"

# Existing Config C is READ-ONLY: only its diagnostics are used to determine matched compute.
CONFIG_C_DIR = "/content/drive/MyDrive/Data_Khoaluan/ppo_config_c_v1"

# New output directory: this notebook never writes into RL_configa.
FLAT_OUTPUT_DIR = "/content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1"
os.makedirs(FLAT_OUTPUT_DIR, exist_ok=True)

SEED_LIST = [42, 123, 777]

print("DATA_DIR      =", DATA_DIR)
print("T5_SFT_DIR    =", T5_SFT_DIR)
print("CONFIG_C_DIR  =", CONFIG_C_DIR)
print("FLAT_OUTPUT   =", FLAT_OUTPUT_DIR)
print("SEEDS         =", SEED_LIST)

DATA_DIR      = /content/drive/MyDrive/Data_Khoaluan/03_stratified_benchmark
T5_SFT_DIR    = /content/drive/MyDrive/Data_Khoaluan/T5_base
CONFIG_C_DIR  = /content/drive/MyDrive/Data_Khoaluan/ppo_config_c_v1
FLAT_OUTPUT   = /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1
SEEDS         = [42, 123, 777]


## 3. Load the new split

In [3]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_toxic = train_df.reset_index(drop=True)
val_toxic   = val_df.reset_index(drop=True)
test_toxic  = test_df.reset_index(drop=True)

print("train =", len(train_toxic))
print("val   =", len(val_toxic))
print("test  =", len(test_toxic))
print("\nTraining band distribution:")
print(train_toxic["band"].value_counts())

assert len(train_toxic) == 2424, f"Expected 2424 filtered training rows, got {len(train_toxic)}"
assert set(train_toxic["band"].str.lower().unique()) == {"easy", "medium", "hard"}

train = 2424
val   = 672
test  = 1312

Training band distribution:
band
easy      1149
medium     754
hard       521
Name: count, dtype: int64


## 4. Load the same reward functions/models as Config C

In [4]:
from transformers import T5TokenizerFast, AutoTokenizer, AutoModelForSequenceClassification
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

tokenizer = T5TokenizerFast.from_pretrained(T5_SFT_DIR)
TASK_PREFIX = 'detoxify: '
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128

def to_list(x):
    if torch.is_tensor(x): return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray): return x.tolist()
    return list(x)

tox_tokenizer = AutoTokenizer.from_pretrained('unitary/unbiased-toxic-roberta')
tox_model = AutoModelForSequenceClassification.from_pretrained('unitary/unbiased-toxic-roberta').to(DEVICE)
tox_model.eval()
id2label = tox_model.config.id2label
TOX_IDX = next((int(i) for i, l in id2label.items() if str(l).lower() in {'toxic','toxicity'}), 0)

@torch.no_grad()
def compute_r_tox(texts):
    enc = tox_tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
    probs = torch.sigmoid(tox_model(**enc).logits.float())[:, TOX_IDX]
    return to_list(1.0 - probs)

sbert_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE)

def compute_r_sem(X_texts, Y_texts):
    eX = sbert_model.encode(X_texts, convert_to_tensor=True, device=DEVICE)
    eY = sbert_model.encode(Y_texts, convert_to_tensor=True, device=DEVICE)
    return to_list((F.cosine_similarity(eX, eY, dim=1) + 1.0) / 2.0)

gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
gpt2_model.eval()
MAX_LOSS_CLAMP = 20.0

@torch.no_grad()
def compute_perplexity(text):
    enc = gpt2_tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    if enc.input_ids.size(1) < 2:
        return 1.0
    loss = torch.clamp(gpt2_model(enc.input_ids, labels=enc.input_ids).loss, max=MAX_LOSS_CLAMP)
    return torch.exp(loss).item()

def compute_r_flu(texts):
    """Standard per-sample evaluation (unbatched)."""
    return [1.0 / (1.0 + math.log(max(compute_perplexity(t), 1.0 + 1e-6)) / 5.0) for t in texts]

def compute_r_len(X_texts, Y_texts):
    rewards = []
    for x, y in zip(X_texts, Y_texts):
        x_len, y_len = max(len(x.split()), 1), len(y.split())
        ratio = y_len / x_len
        if 0.7 <= ratio <= 1.3: r = 1.0
        elif ratio < 0.7: r = y_len / (0.7 * x_len)
        else: r = (1.3 * x_len) / max(y_len, 1)
        rewards.append(r)
    return rewards

POLITENESS_LEXICON = [
    'please', 'could you', 'would you', 'might you', 'can you', 'will you',
    'hi', 'hello', 'hey', 'greetings', 'howdy', 'good morning', 'good evening', 'good afternoon',
    'goodbye', 'bye', 'see you later', 'thank', 'grateful', 'gratitude',
    'sorry', 'woops', 'oops', 'whoops', 'apologize',
    'is okay', 'not worry', 'no big deal', 'not a big deal', 'no problem', 'no worries',
    'is fine', 'you are good', "it's fine", "it's okay", 'let me know', 'for you',
]
P_SIZE = len(POLITENESS_LEXICON)

def compute_r_pol(Y_texts):
    return [min(sum(1 for w in POLITENESS_LEXICON if w in y.lower()) / P_SIZE, 1.0) for y in Y_texts]

def compute_r_total(X_texts, Y_texts, weights):
    r_tox, r_sem, r_flu = compute_r_tox(Y_texts), compute_r_sem(X_texts, Y_texts), compute_r_flu(Y_texts)
    r_len, r_pol = compute_r_len(X_texts, Y_texts), compute_r_pol(Y_texts)
    total = [weights['tox']*r_tox[i] + weights['sem']*r_sem[i] + weights['flu']*r_flu[i]
             + weights['len']*r_len[i] + weights['pol']*r_pol[i] for i in range(len(Y_texts))]
    return total, {'r_tox':r_tox,'r_sem':r_sem,'r_flu':r_flu,'r_len':r_len,'r_pol':r_pol}

logger.info('Reward functions initialized successfully.')

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 5. Same PPO configuration as Config C

In [5]:
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model

BETA_KL, CLIP_EPS, GAE_GAMMA, GAE_LAMBDA = 0.05, 0.2, 0.99, 0.95
LEARNING_RATE, PPO_BATCH_SIZE, PPO_MINI_BATCH_SIZE = 1.4e-5, 8, 4

ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE, batch_size=PPO_BATCH_SIZE, mini_batch_size=PPO_MINI_BATCH_SIZE,
    ppo_epochs=4, init_kl_coef=BETA_KL, cliprange=CLIP_EPS, cliprange_value=CLIP_EPS,
    gamma=GAE_GAMMA, lam=GAE_LAMBDA,
)
logger.info('PPOConfig ready.')

/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


## 6. Flat reward and matched-compute definition

In [6]:
# Config C Phase-3 reward is FIXED for the entire flat run.
FLAT_WEIGHTS = {
    "tox": 0.15,
    "sem": 0.30,
    "flu": 0.20,
    "len": 0.25,
    "pol": 0.10,
}

MAX_NEW_TOKENS = 128
CHECKPOINT_EVERY = 50

def actual_config_c_steps(seed):
    phase_paths = [
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase1_easy.csv"),
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase2_balance.csv"),
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase3_refinement.csv"),
    ]

    counts = []
    for p in phase_paths:
        if not os.path.exists(p):
            print(f" Warning: Missing file {p} -> Hardcoding step = 50.")
            counts.append(50)
        else:
            try:
                df = pd.read_csv(p)
                if "step" not in df.columns:
                    counts.append(50)
                else:
                    counts.append(len(df))
            except Exception:
                counts.append(50)

    return counts, sum(counts)

for seed in SEED_LIST:
    phase_counts, total = actual_config_c_steps(seed)
    print(f"Seed {seed}: Phase1={phase_counts[0]}, Phase2={phase_counts[1]}, "
          f"Phase3={phase_counts[2]} -> Flat target={total} updates")

print("\nFlat reward:", FLAT_WEIGHTS)
print("No Easy/Medium/Hard reward switching occurs during Flat PPO.")

Seed 42: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates
Seed 123: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates
Seed 777: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates

Flat reward: {'tox': 0.15, 'sem': 0.3, 'flu': 0.2, 'len': 0.25, 'pol': 0.1}
No Easy/Medium/Hard reward switching occurs during Flat PPO.


## 7. Flat PPO training loop

In [7]:
from transformers import T5TokenizerFast
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model

def safe_generate_flat(ppo_trainer, query_tensors, max_attempts=5):
    """
    Uses the ORIGINAL Config-A generation settings.
    If a response is empty/all-padding, regenerate the affected batch rather than
    changing generation parameters.
    """
    last_responses = None

    for attempt in range(max_attempts):
        response_tensors = ppo_trainer.generate(
            query_tensors,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            top_k=0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
        last_responses = response_tensors

        bad = []
        for j, r in enumerate(response_tensors):
            if r.numel() == 0:
                bad.append(j)
                continue
            # Remove padding only for the validity check; generation config is unchanged.
            valid = r[r != tokenizer.pad_token_id]
            if valid.numel() == 0:
                bad.append(j)

        if not bad:
            return response_tensors

        logger.warning(
            f"Empty/all-padding response detected (attempt {attempt+1}/{max_attempts}); "
            "regenerating with the same generation configuration."
        )

    raise RuntimeError(
        "Could not obtain a non-empty PPO response after retries. "
        "Training stopped rather than silently changing the generation configuration."
    )

def run_flat_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    # Match the actual number of PPO updates performed by Config C for this seed.
    phase_counts, target_steps = actual_config_c_steps(seed)

    # Full training corpus: Easy + Medium + Hard pooled from the beginning.
    texts_all = train_toxic["text"].astype(str).tolist()
    assert len(texts_all) == 2424

    # Resume-safe: continue from a saved flat checkpoint if available.
    resume_candidates = sorted(
        glob.glob(os.path.join(seed_dir, "flat_step_*")),
        key=lambda p: int(re.search(r"flat_step_(\d+)$", p).group(1))
    )
    if resume_candidates:
        resume_dir = resume_candidates[-1]
        m = re.search(r"flat_step_(\d+)$", resume_dir)
        start_step = int(m.group(1))
        policy_dir = resume_dir
        print(f"[Seed {seed}] Resuming from {policy_dir} at step {start_step}/{target_steps}")
    else:
        start_step = 0
        policy_dir = T5_SFT_DIR
        print(f"[Seed {seed}] Starting from Full-SFT checkpoint.")

    policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(policy_dir).to(DEVICE)
    ref = create_reference_model(policy).to(DEVICE)
    trainer = PPOTrainer(config=ppo_config, model=policy, ref_model=ref, tokenizer=tokenizer)

    diagnostics_path = os.path.join(seed_dir, "flat_diagnostics.csv")
    diagnostics = []
    if os.path.exists(diagnostics_path):
        old = pd.read_csv(diagnostics_path)
        diagnostics = old.to_dict("records")

    step = start_step
    t0 = time.time()

    while step < target_steps:
        idx = random.sample(range(len(texts_all)), PPO_BATCH_SIZE)
        X_batch = [texts_all[i] for i in idx]

        query_tensors = [
            tokenizer(
                TASK_PREFIX + x,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SOURCE_LEN
            ).input_ids[0].to(DEVICE)
            for x in X_batch
        ]

        response_tensors = safe_generate_flat(trainer, query_tensors)
        Y_batch = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

        rewards_list, _ = compute_r_total(X_batch, Y_batch, FLAT_WEIGHTS)
        reward_tensors = [torch.tensor(r, dtype=torch.float32) for r in rewards_list]

        stats = trainer.step(query_tensors, response_tensors, reward_tensors)
        step += 1

        row = {
            "step": step,
            "mean_reward": float(np.mean(rewards_list)),
            "kl": stats.get("objective/kl"),
            "policy_loss": stats.get("ppo/loss/policy"),
        }
        diagnostics.append(row)

        if step % 5 == 0:
            print(
                f"[Seed {seed}] step={step}/{target_steps} "
                f"R={row['mean_reward']:.4f} KL={row['kl']} "
                f"policy_loss={row['policy_loss']}"
            )

        if step % CHECKPOINT_EVERY == 0 or step == target_steps:
            ckpt_dir = os.path.join(seed_dir, f"flat_step_{step}")
            policy.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            pd.DataFrame(diagnostics).to_csv(diagnostics_path, index=False)
            print(f"[Seed {seed}] checkpoint saved: {ckpt_dir}")

    # Final manifest for this seed
    manifest = {
        "seed": seed,
        "target_steps": target_steps,
        "config_a_phase_steps": {
            "phase1_easy": phase_counts[0],
            "phase2_balance": phase_counts[1],
            "phase3_refinement": phase_counts[2],
        },
        "train_rows": len(texts_all),
        "reward_weights": FLAT_WEIGHTS,
        "generation": {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": True,
            "top_k": 0,
            "top_p": 1.0,
            "pad_token_id": tokenizer.pad_token_id,
        },
        "ppo": {
            "learning_rate": LEARNING_RATE,
            "batch_size": PPO_BATCH_SIZE,
            "mini_batch_size": PPO_MINI_BATCH_SIZE,
            "ppo_epochs": 4,
            "kl_coef": BETA_KL,
            "cliprange": CLIP_EPS,
            "gamma": GAE_GAMMA,
            "gae_lambda": GAE_LAMBDA,
        },
        "flat_definition": "Easy+Medium+Hard pooled from the beginning; no curriculum phases.",
    }
    with open(os.path.join(seed_dir, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    del trainer, ref, policy
    gc.collect()
    torch.cuda.empty_cache()

    print(f"[Seed {seed}] DONE — {step} updates.")
    return step

## 8. Run all three flat seeds

In [8]:
for seed in SEED_LIST:
    run_flat_seed(seed)

[Seed 42] Starting from Full-SFT checkpoint.


Loading weights:   0%|          | 0/257 [00:01<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.py:273: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[Seed 42] step=5/140 R=0.7021 KL=0.46504729986190796 policy_loss=-0.03235314041376114
[Seed 42] step=10/140 R=0.6611 KL=0.6476275324821472 policy_loss=-0.06119761988520622
[Seed 42] step=15/140 R=0.5754 KL=1.685563564300537 policy_loss=-0.03335084766149521
[Seed 42] step=20/140 R=0.6328 KL=1.0432721376419067 policy_loss=-0.04913293197751045
[Seed 42] step=25/140 R=0.6428 KL=1.3408108949661255 policy_loss=-0.026644758880138397
[Seed 42] step=30/140 R=0.6463 KL=1.9724538326263428 policy_loss=0.023659199476242065
[Seed 42] step=35/140 R=0.6275 KL=0.709345817565918 policy_loss=-0.03606613725423813
[Seed 42] step=40/140 R=0.6916 KL=1.3431792259216309 policy_loss=-0.06484630703926086
[Seed 42] step=45/140 R=0.6264 KL=2.109391689300537 policy_loss=-0.05764874070882797
[Seed 42] step=50/140 R=0.6486 KL=0.6791151165962219 policy_loss=-0.029763799160718918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

[Seed 777] checkpoint saved: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_777/flat_step_100


/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.py:1246: UserWarning: The average ratio of batch (11.41) exceeds threshold 10.00. Skipping batch.
  warnings.warn(


[Seed 777] step=105/140 R=0.6570 KL=-0.07217007875442505 policy_loss=0.021344147622585297
[Seed 777] step=110/140 R=0.6122 KL=0.9621255397796631 policy_loss=-0.020016884431242943
[Seed 777] step=115/140 R=0.6432 KL=2.188113212585449 policy_loss=-0.012261012569069862
[Seed 777] step=120/140 R=0.6252 KL=1.5314881801605225 policy_loss=-0.030087381601333618
[Seed 777] step=125/140 R=0.6928 KL=0.8318208456039429 policy_loss=-0.02566193789243698
[Seed 777] step=130/140 R=0.6648 KL=1.5274337530136108 policy_loss=-0.023562071844935417
[Seed 777] step=135/140 R=0.6817 KL=0.6303728222846985 policy_loss=-0.02729477733373642
[Seed 777] step=140/140 R=0.6803 KL=1.989358901977539 policy_loss=-0.02648337185382843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[Seed 777] checkpoint saved: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_777/flat_step_140
[Seed 777] DONE — 140 updates.


## 9. Final verification

In [9]:
print("===== FLAT PPO FINAL CHECK =====")
for seed in SEED_LIST:
    seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
    phase_counts, target = actual_config_c_steps(seed)
    diag = os.path.join(seed_dir, "flat_diagnostics.csv")

    if os.path.exists(diag):
        df = pd.read_csv(diag)
        actual = len(df)
        print(
            f"Seed {seed}: flat={actual} updates | "
            f"target={target} | Config C phases={phase_counts}"
        )
        assert actual == target, f"Seed {seed}: update mismatch"
    else:
        print(f"Seed {seed}: diagnostics missing")

print("\nOutput directory:", FLAT_OUTPUT_DIR)

===== FLAT PPO FINAL CHECK =====
Seed 42: flat=140 updates | target=140 | Config C phases=[40, 40, 60]
Seed 123: flat=140 updates | target=140 | Config C phases=[40, 40, 60]
Seed 777: flat=140 updates | target=140 | Config C phases=[40, 40, 60]

Output directory: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1


## Important design choice

The original Config C uses different reward vectors across its curriculum phases.  
For the **flat/mixed control**, the reward vector is fixed to **Config C Phase 3**:

\[
(\lambda_{\mathrm{tox}},\lambda_{\mathrm{sem}},\lambda_{\mathrm{flu}},
\lambda_{\mathrm{len}},\lambda_{\mathrm{pol}})
=(0.30,0.30,0.20,0.10,0.10).
\]

This notebook therefore changes the **curriculum/data exposure schedule** while keeping the final Config-A reward configuration fixed. It does **not** assign Easy/Medium/Hard-specific rewards to individual samples.

The notebook also does not use early stopping for Flat PPO, because the control must match the actual total PPO updates of Config C for each seed.

## RUN EVALUATION (INFERENCE) ON CHECKPOINTS

In [10]:
TAU_REDETECT, SEM_THRESHOLD, FLU_THRESHOLD, MAX_REGEN_ATTEMPTS = 0.30, 0.70, 0.70, 3
TEMPERATURES = [0.7, 0.9, 1.1]

def generate_one_with_temp(model, x_text, temperature):
    inputs = tokenizer(TASK_PREFIX+x_text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                   temperature=temperature, top_k=0, top_p=1.0)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

def passes_post_validation(x_text, y_text):
    t_out = 1.0 - compute_r_tox([y_text])[0]
    r_sem = compute_r_sem([x_text], [y_text])[0]
    r_flu = compute_r_flu([y_text])[0]
    y_len, x_len = len(y_text.split()), max(len(x_text.split()), 1)
    ratio = y_len / x_len
    return (t_out < TAU_REDETECT) and (r_sem > SEM_THRESHOLD) and (r_flu > FLU_THRESHOLD) and (0.7 <= ratio <= 1.3)

def generate_with_post_validation(model, X_texts):
    """Standard per-sample evaluation (unbatched)."""
    Y_out, n_failed = [], 0
    for x in X_texts:
        passed, last_y = False, None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            y = generate_one_with_temp(model, x, TEMPERATURES[attempt])
            last_y = y
            if passes_post_validation(x, y):
                passed = True
                break
        if not passed:
            n_failed += 1
        Y_out.append(last_y)
    return Y_out, n_failed

def evaluate_checkpoint(model, eval_df, use_post_validation=False):
    X_texts, labels, bands = eval_df['text'].tolist(), eval_df['label'].tolist(), eval_df['band'].tolist()
    if use_post_validation:
        Y_texts, n_failed = generate_with_post_validation(model, X_texts)
    else:
        Y_texts, n_failed = generate_batch_deterministic(model, X_texts), None
    STA = compute_r_tox(Y_texts)
    SIM = compute_r_sem(X_texts, Y_texts)
    FLUE = compute_r_flu(Y_texts)
    Joint = [(s*si*f)**(1/3) for s,si,f in zip(STA,SIM,FLUE)]
    df = pd.DataFrame({'X':X_texts,'Y':Y_texts,'label':labels,'band':bands,'STA':STA,'Sim':SIM,'Flu':FLUE,'Joint':Joint})
    return df, n_failed

def summarize(df):
    return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

logger.info('Post-Validation (standard) + evaluate_checkpoint/summarize ready.')

In [11]:
EVAL_BATCH_SIZE = 16
TAU_REDETECT, SEM_THRESHOLD, FLU_THRESHOLD, MAX_REGEN_ATTEMPTS = 0.30, 0.70, 0.70, 3
TEMPERATURES = [0.7, 0.9, 1.1]

def generate_batch_deterministic(model, texts, batch_size=EVAL_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    results = []
    n_batches = math.ceil(len(texts) / batch_size)
    for bi in range(n_batches):
        batch_texts = texts[bi*batch_size:(bi+1)*batch_size]
        inputs = [TASK_PREFIX + t for t in batch_texts]
        try:
            enc = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
            with torch.no_grad():
                out_ids = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
            results.extend([tokenizer.decode(o, skip_special_tokens=True) for o in out_ids])
        except RuntimeError as e:
            logger.error(f'Loi batch {bi}: {e}. Thu tung cau.')
            for t in batch_texts:
                try:
                    enc1 = tokenizer(TASK_PREFIX+t, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
                    with torch.no_grad():
                        o = model.generate(**enc1, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
                    results.append(tokenizer.decode(o[0], skip_special_tokens=True))
                except Exception as e2:
                    logger.error(f'Fail 1 cau: {e2}')
                    results.append(t)
    return results

def generate_one_with_temp(model, x_text, temperature):
    inputs = tokenizer(TASK_PREFIX+x_text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                   temperature=temperature, top_k=0, top_p=1.0)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

def passes_post_validation(x_text, y_text):
    t_out = 1.0 - compute_r_tox([y_text])[0]
    r_sem = compute_r_sem([x_text], [y_text])[0]
    r_flu = compute_r_flu([y_text])[0]
    y_len, x_len = len(y_text.split()), max(len(x_text.split()), 1)
    ratio = y_len / x_len
    return (t_out < TAU_REDETECT) and (r_sem > SEM_THRESHOLD) and (r_flu > FLU_THRESHOLD) and (0.7 <= ratio <= 1.3)

def generate_with_post_validation(model, X_texts):
    Y_out, n_failed = [], 0
    for x in X_texts:
        passed, last_y = False, None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            y = generate_one_with_temp(model, x, TEMPERATURES[attempt])
            last_y = y
            if passes_post_validation(x, y):
                passed = True
                break
        if not passed:
            n_failed += 1
        Y_out.append(last_y)
    return Y_out, n_failed

def evaluate_checkpoint(model, eval_df, use_post_validation=False):
    X_texts, labels, bands = eval_df['text'].tolist(), eval_df['label'].tolist(), eval_df['band'].tolist()
    if use_post_validation:
        Y_texts, n_failed = generate_with_post_validation(model, X_texts)
    else:
        Y_texts, n_failed = generate_batch_deterministic(model, X_texts), None
    STA  = compute_r_tox(Y_texts)
    SIM  = compute_r_sem(X_texts, Y_texts)
    FLUE = compute_r_flu(Y_texts)
    Joint = [(s*si*f)**(1/3) for s, si, f in zip(STA, SIM, FLUE)]
    df = pd.DataFrame({'X':X_texts,'Y':Y_texts,'label':labels,'band':bands,
                       'STA':STA,'Sim':SIM,'Flu':FLUE,'Joint':Joint})
    return df, n_failed

def summarize(df):
    return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),
            'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

logger.info('generate_batch_deterministic + evaluate_checkpoint + summarize san sang.')

import gc

RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=== EVALUATING FLAT PPO CHECKPOINTS ===")

for seed in SEED_LIST:
    print(f"\n--- Evaluating Seed {seed} ---")
    out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')

    if os.path.exists(out_csv):
        print(f" Results already exist for Seed {seed} at {out_csv}, skipping generation.")
        continue

    seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")

    # Find latest checkpoint (highest step)
    import glob, re
    ckpt_dirs = glob.glob(os.path.join(seed_dir, "flat_step_*"))
    if not ckpt_dirs:
        print(f" No checkpoint found for Seed {seed}")
        continue

    best_ckpt = sorted(ckpt_dirs, key=lambda p: int(re.search(r'flat_step_(\d+)', p).group(1)))[-1]
    print(f"Loading checkpoint: {best_ckpt}")

    m_flat = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(best_ckpt).to(DEVICE)
    m_flat.eval()

    print("Starting generation and evaluation...")
    per_sample_df, _ = evaluate_checkpoint(m_flat, test_toxic, use_post_validation=False)
    per_sample_df.to_csv(out_csv, index=False)
    print(f" Saved results for Seed {seed} successfully.")

    del m_flat
    gc.collect()
    torch.cuda.empty_cache()

print("\n COMPLETED EVALUATION (INFERENCE) FOR 3 SEEDS!")


=== EVALUATING FLAT PPO CHECKPOINTS ===

--- Evaluating Seed 42 ---
Loading checkpoint: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_42/flat_step_140


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_42/flat_step_140
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.bias   | UNEXPECTED |  | 
v_head.summary.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting generation and evaluation...
 Saved results for Seed 42 successfully.

--- Evaluating Seed 123 ---
Loading checkpoint: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_123/flat_step_140


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_123/flat_step_140
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.bias   | UNEXPECTED |  | 
v_head.summary.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting generation and evaluation...
 Saved results for Seed 123 successfully.

--- Evaluating Seed 777 ---
Loading checkpoint: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_777/flat_step_140


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_777/flat_step_140
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.bias   | UNEXPECTED |  | 
v_head.summary.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting generation and evaluation...
 Saved results for Seed 777 successfully.

 COMPLETED EVALUATION (INFERENCE) FOR 3 SEEDS!


## EVALUATION SUMMARY RESULTS (MEAN ± STD)

In [12]:
import pandas as pd
import numpy as np
import os

RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
# Create output directory if not exists
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED_LIST = [42, 123, 777]

def summarize(df):
    return {
        'STA': float(df['STA'].mean()),
        'Sim': float(df['Sim'].mean()),
        'Flu': float(df['Flu'].mean()),
        'Joint': float(df['Joint'].mean())
    }

records = []

for seed in SEED_LIST:
    # Read FLAT Phase results
    csv_path = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
    if not os.path.exists(csv_path):
        print(f" Evaluation file not found: {csv_path}. Please run evaluation cells above first!")
        continue

    df = pd.read_csv(csv_path)

    # Compute for 3 groups
    full_metrics = summarize(df)
    toxic_metrics = summarize(df[df['label'] == 'toxic'])
    nontoxic_metrics = summarize(df[df['label'] == 'nontoxic'])

    records.append({
        'Seed': seed,
        'Group': 'FULL',
        'STA': full_metrics['STA'], 'Sim': full_metrics['Sim'], 'Flu': full_metrics['Flu'], 'Joint': full_metrics['Joint']
    })
    records.append({
        'Seed': seed,
        'Group': 'TOXIC',
        'STA': toxic_metrics['STA'], 'Sim': toxic_metrics['Sim'], 'Flu': toxic_metrics['Flu'], 'Joint': toxic_metrics['Joint']
    })
    records.append({
        'Seed': seed,
        'Group': 'NONTOXIC',
        'STA': nontoxic_metrics['STA'], 'Sim': nontoxic_metrics['Sim'], 'Flu': nontoxic_metrics['Flu'], 'Joint': nontoxic_metrics['Joint']
    })

if len(records) > 0:
    results_df = pd.DataFrame(records)
    print(" RESULTS PER SEED (PHASE FLAT):")
    display(results_df.round(4))

    # Compute Mean and Std Dev across 3 seeds
    summary_df = results_df.groupby('Group').agg(['mean', 'std']).reset_index()

    # Format as Mean ± Std
    formatted_records = []
    for _, row in summary_df.iterrows():
        grp = row['Group'].values[0] if isinstance(row['Group'], pd.Series) else row['Group'][0]
        formatted_records.append({
            'Group': grp,
            'STA': f"{row[('STA','mean')]:.3f} ± {row[('STA','std')]:.3f}",
            'Sim': f"{row[('Sim','mean')]:.3f} ± {row[('Sim','std')]:.3f}",
            'Flu': f"{row[('Flu','mean')]:.3f} ± {row[('Flu','std')]:.3f}",
            'Joint': f"{row[('Joint','mean')]:.3f} ± {row[('Joint','std')]:.3f}",
        })

    final_table = pd.DataFrame(formatted_records)
    # Reorder FULL -> TOXIC -> NONTOXIC
    final_table['Group'] = pd.Categorical(final_table['Group'], categories=['FULL', 'TOXIC', 'NONTOXIC'], ordered=True)
    final_table = final_table.sort_values('Group').reset_index(drop=True)

    print("\n SUMMARY TABLE MEAN ± STD (FLAT PPO):")
    display(final_table)
else:
    print(" No data available for aggregation.")


 RESULTS PER SEED (PHASE FLAT):


,Seed,Group,STA,Sim,Flu,Joint
0,42,FULL,0.8811,0.9341,0.5382,0.7470
1,42,TOXIC,0.8780,0.9329,0.5395,0.7454
2,42,NONTOXIC,0.9026,0.9429,0.5291,0.7579
3,123,FULL,0.8947,0.9181,0.5329,0.7454
4,123,TOXIC,0.8899,0.9174,0.5340,0.7433
5,123,NONTOXIC,0.9287,0.9237,0.5247,0.7597
6,777,FULL,0.8929,0.9265,0.5387,0.7510
7,777,TOXIC,0.8909,0.9244,0.5399,0.7497
8,777,NONTOXIC,0.9066,0.9410,0.5303,0.7600



 SUMMARY TABLE MEAN ± STD (FLAT PPO):


,Group,STA,Sim,Flu,Joint
0,FULL,0.890 ± 0.007,0.926 ± 0.008,0.537 ± 0.003,0.748 ± 0.003
1,TOXIC,0.886 ± 0.007,0.925 ± 0.008,0.538 ± 0.003,0.746 ± 0.003
2,NONTOXIC,0.913 ± 0.014,0.936 ± 0.011,0.528 ± 0.003,0.759 ± 0.001


## BEHAVIOR AND PPL METRICS

In [13]:
import pandas as pd
import numpy as np
import os
import math

def tokenize_simple(text): return str(text).lower().split()

def levenshtein_token(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[n][m]

def compute_edit_behavior(row):
    x, y = str(row['X']).strip(), str(row['Y']).strip()
    xt, yt = tokenize_simple(x), tokenize_simple(y)
    xl = max(len(xt),1); yl = len(yt)
    return pd.Series({'unchanged': x.lower()==y.lower(), 'token_edit_rate': levenshtein_token(xt,yt)/xl,
                        'truncated': yl<xl, 'clause_loss': ((xl-yl)/xl) > 0.30})

_all = []
RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
SEED_LIST = [42, 123, 777]

# 1. PER-SEED TABLE
print('=== 1. BEHAVIOR AND PPL PER SEED ===')
seed_results = []
seed_ppl = []

for seed in SEED_LIST:
    p = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
    if os.path.exists(p):
        df = pd.read_csv(p)
        df['seed'] = seed
        df['phase'] = 'phase_flat'
        _all.append(df)

        # Compute behavior metrics
        m = df.apply(compute_edit_behavior, axis=1)
        seed_results.append({'Seed':seed, 'n':len(df),
                          'Unchanged (%)':round(m['unchanged'].mean()*100,1),
                          'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                          'Short-fail (%)':round(m['truncated'].mean()*100,1),
                          'Truncation >30% (%)':round(m['clause_loss'].mean()*100,1)})

        # Compute PPL
        ppls = pd.Series([compute_perplexity(str(y)) for y in df['Y']])
        pct = ((ppls>=20)&(ppls<=150)).mean()
        seed_ppl.append({'Seed':seed, 'mean_PPL':round(ppls.mean(),1), 'trong[20,150]':round(pct*100,1)})

if not _all:
    print(" No evaluation CSV files found in final_results.")
else:
    print("\n[Behavior Per Seed - FULL]")
    display(pd.DataFrame(seed_results))
    print("\n[PPL Per Seed - FULL]")
    display(pd.DataFrame(seed_ppl))

    # 2. POOLED TABLE SIMILAR TO CONFIG A
    all_per_sample_pooled = pd.concat(_all, ignore_index=True)

    print('\n\n=== 2. POOLED TABLES ===')

    print('\n[4 BEHAVIOR METRICS TABLE — FULL]')
    results = []
    m = all_per_sample_pooled.apply(compute_edit_behavior, axis=1)
    results.append({'Phase':'phase_flat (Pooled)','n':len(all_per_sample_pooled),
                      'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                      'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                      'Short-fail Rate (%)':round(m['truncated'].mean()*100,1),
                      'Truncation >30% Rate (%)':round(m['clause_loss'].mean()*100,1)})
    edit_behavior_full = pd.DataFrame(results)
    print(edit_behavior_full.to_string(index=False))
    edit_behavior_full.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_full.csv'), index=False)

    print('\n[4 BEHAVIOR METRICS TABLE — Toxic-only]')
    results_toxic = []
    _pooled_toxic_behavior = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
    m = _pooled_toxic_behavior.apply(compute_edit_behavior, axis=1)
    results_toxic.append({'Phase':'phase_flat (Pooled)','n':len(_pooled_toxic_behavior),
                      'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                      'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                      'Short-fail Rate (%)':round(m['truncated'].mean()*100,1),
                      'Truncation >30% Rate (%)':round(m['clause_loss'].mean()*100,1)})
    edit_behavior_toxic = pd.DataFrame(results_toxic)
    print(edit_behavior_toxic.to_string(index=False))
    edit_behavior_toxic.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_toxic_only.csv'), index=False)

    print('\n[PPL TABLE — FULL]')
    ppl_rows = []
    ppls = pd.Series([compute_perplexity(str(y)) for y in all_per_sample_pooled['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows.append({'phase':'phase_flat (Pooled)','mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f"phase_flat (Pooled): mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}")
    ppl_full = pd.DataFrame(ppl_rows)
    ppl_full.to_csv(os.path.join(RESULTS_DIR, 'ppl_full.csv'), index=False)

    print('\n[PPL TABLE — Toxic-only]')
    ppl_rows_toxic = []
    _pooled_toxic = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
    ppls = pd.Series([compute_perplexity(str(y)) for y in _pooled_toxic['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows_toxic.append({'phase':'phase_flat (Pooled)','mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f"phase_flat (Pooled): mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}")
    ppl_toxic = pd.DataFrame(ppl_rows_toxic)
    ppl_toxic.to_csv(os.path.join(RESULTS_DIR, 'ppl_toxic_only.csv'), index=False)


=== 1. BEHAVIOR AND PPL PER SEED ===

[Behavior Per Seed - FULL]


,Seed,n,Unchanged (%),Token Edit Rate (%),Short-fail (%),Truncation >30% (%)
0,42,1312,3.4,41.7,93.4,64.9
1,123,1312,0.1,48.1,94.1,73.0
2,777,1312,3.5,45.0,93.4,71.8



[PPL Per Seed - FULL]


,Seed,mean_PPL,"trong[20,150]"
0,42,120.3,77.1
1,123,143.5,73.4
2,777,122.2,77.3




=== 2. POOLED TABLES ===

[4 BEHAVIOR METRICS TABLE — FULL]
              Phase    n  Unchanged Rate (%)  Token Edit Rate (%)  Short-fail Rate (%)  Truncation >30% Rate (%)
phase_flat (Pooled) 3936                  2.3                    44.9               93.7                   69.9

[4 BEHAVIOR METRICS TABLE — Toxic-only]
              Phase    n  Unchanged Rate (%)  Token Edit Rate (%)  Short-fail Rate (%)  Truncation >30% Rate (%)
phase_flat (Pooled) 3450                  2.1                    46.1               95.0                   71.9

[PPL TABLE — FULL]
phase_flat (Pooled): mean=  128.7, trong[20,150]=75.9%

[PPL TABLE — Toxic-only]
phase_flat (Pooled): mean=  128.9, trong[20,150]=76.2%


## Step E1 — Evaluate: Load checkpoints & generate outputs (inference only, no training)

In [14]:
import gc, glob, re
from trl import AutoModelForSeq2SeqLMWithValueHead

RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Baseline SFT
BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')
if os.path.exists(BASELINE_CSV):
    print("Baseline already exists, loading...")
    baseline_per_sample = pd.read_csv(BASELINE_CSV)
else:
    print("Generating baseline (SFT)...")
    baseline_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(T5_SFT_DIR).to(DEVICE)
    baseline_model.eval()
    baseline_per_sample, _ = evaluate_checkpoint(baseline_model, test_toxic, use_post_validation=False)
    baseline_per_sample.to_csv(BASELINE_CSV, index=False)
    del baseline_model
    gc.collect()
    torch.cuda.empty_cache()

baseline_metrics_full     = summarize(baseline_per_sample)
baseline_metrics_toxic    = summarize(baseline_per_sample[baseline_per_sample['label']=='toxic'])
baseline_metrics_nontoxic = summarize(baseline_per_sample[baseline_per_sample['label']=='nontoxic'])
print(f"Baseline | FULL Joint={baseline_metrics_full['Joint']:.3f} | TOXIC Joint={baseline_metrics_toxic['Joint']:.3f} | NONTOXIC Joint={baseline_metrics_nontoxic['Joint']:.3f}")

# PPO Flat checkpoints
for seed in SEED_LIST:
    out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
    if os.path.exists(out_csv):
        print(f"[Seed {seed}] Flat result already exists, skipping.")
        continue

    seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
    ckpt_dirs = sorted(glob.glob(os.path.join(seed_dir, "flat_step_*")),
                       key=lambda p: int(re.search(r'flat_step_(\d+)', p).group(1)))
    if not ckpt_dirs:
        print(f"[Seed {seed}] No checkpoint found! Skipping.")
        continue

    best_ckpt = ckpt_dirs[-1]
    print(f"[Seed {seed}] Loading: {best_ckpt}")
    m = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(best_ckpt).to(DEVICE)
    m.eval()
    df, _ = evaluate_checkpoint(m, test_toxic, use_post_validation=False)
    df.to_csv(out_csv, index=False)
    print(f"[Seed {seed}] Saved: {out_csv}")
    del m
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll inference done!")


Generating baseline (SFT)...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Baseline | FULL Joint=0.759 | TOXIC Joint=0.758 | NONTOXIC Joint=0.764
[Seed 42] Flat result already exists, skipping.
[Seed 123] Flat result already exists, skipping.
[Seed 777] Flat result already exists, skipping.

All inference done!


## Step E2 — Per-seed tables: table3/3b/3c + table4/4b/4c (FULL / Toxic / Non-toxic)

In [15]:
def summarize(df):
    return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

METRICS = ['STA', 'Sim', 'Flu', 'Joint']

for seed in SEED_LIST:
    fname = f'per_sample_seed{seed}_phase_flat.csv'
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[Seed {seed}] Missing {fpath}, skipping.")
        continue

    df = pd.read_csv(fpath)

    t3  = [{'Phase': 'Baseline (SFT)', **baseline_metrics_full},
            {'Phase': 'Flat PPO',      **summarize(df)}]
    t3b = [{'Phase': 'Baseline (SFT)', **baseline_metrics_toxic},
            {'Phase': 'Flat PPO',      **summarize(df[df['label']=='toxic'])}]
    t3c = [{'Phase': 'Baseline (SFT)', **baseline_metrics_nontoxic},
            {'Phase': 'Flat PPO',      **summarize(df[df['label']=='nontoxic'])}]

    # Add Gain column
    def add_gain(rows, base_joint):
        for r in rows:
            r['Gain'] = 0.0 if r['Phase']=='Baseline (SFT)' else round((r['Joint']-base_joint)/base_joint*100, 2)
        return rows

    t3  = add_gain(t3,  baseline_metrics_full['Joint'])
    t3b = add_gain(t3b, baseline_metrics_toxic['Joint'])
    t3c = add_gain(t3c, baseline_metrics_nontoxic['Joint'])

    pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{seed}.csv'), index=False)
    pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{seed}.csv'), index=False)
    pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{seed}.csv'), index=False)

    print(f'=== SEED {seed} --- TABLE 3 (FULL) ===')
    print(pd.DataFrame(t3).round(3).to_string(index=False))
    print(f'\n=== SEED {seed} --- TABLE 3b (Toxic) ===')
    print(pd.DataFrame(t3b).round(3).to_string(index=False))
    print(f'\n=== SEED {seed} --- TABLE 3c (Non-toxic) ===')
    print(pd.DataFrame(t3c).round(3).to_string(index=False))

    # Table 4/4b/4c by difficulty band
    t4, t4b, t4c = [], [], []
    for band in ['easy', 'medium', 'hard']:
        bf = df[df['band']==band]
        bt = df[(df['band']==band)&(df['label']=='toxic')]
        bn = df[(df['band']==band)&(df['label']=='nontoxic')]
        if len(bf)>0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize(bf)})
        if len(bt)>0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize(bt)})
        if len(bn)>0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize(bn)})

    t4.append({'Difficulty':'Overall','Count':len(df), **summarize(df)})
    t4b.append({'Difficulty':'Overall','Count':len(df[df['label']=='toxic']), **summarize(df[df['label']=='toxic'])})
    t4c.append({'Difficulty':'Overall','Count':len(df[df['label']=='nontoxic']), **summarize(df[df['label']=='nontoxic'])})

    pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{seed}.csv'), index=False)
    pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{seed}.csv'), index=False)
    pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{seed}.csv'), index=False)

    print(f'\n=== SEED {seed} --- TABLE 4 (FULL, band) ===')
    print(pd.DataFrame(t4).round(3).to_string(index=False))
    print(f'\n=== SEED {seed} --- TABLE 4b (Toxic, band) ===')
    print(pd.DataFrame(t4b).round(3).to_string(index=False))
    print(f'\n=== SEED {seed} --- TABLE 4c (Non-toxic, band) ===')
    print(pd.DataFrame(t4c).round(3).to_string(index=False))
    print()


=== SEED 42 --- TABLE 3 (FULL) ===
         Phase   STA   Sim   Flu  Joint  Gain
Baseline (SFT) 0.867 0.972 0.548  0.759  0.00
      Flat PPO 0.881 0.934 0.538  0.747 -1.56

=== SEED 42 --- TABLE 3b (Toxic) ===
         Phase   STA   Sim  Flu  Joint  Gain
Baseline (SFT) 0.866 0.971 0.55  0.758  0.00
      Flat PPO 0.878 0.933 0.54  0.745 -1.66

=== SEED 42 --- TABLE 3c (Non-toxic) ===
         Phase   STA   Sim   Flu  Joint  Gain
Baseline (SFT) 0.875 0.980 0.538  0.764  0.00
      Flat PPO 0.903 0.943 0.529  0.758 -0.83

=== SEED 42 --- TABLE 4 (FULL, band) ===
Difficulty  Count   STA   Sim   Flu  Joint
      Easy    636 0.920 0.936 0.544  0.771
    Medium    379 0.882 0.940 0.531  0.745
      Hard    297 0.798 0.922 0.535  0.698
   Overall   1312 0.881 0.934 0.538  0.747

=== SEED 42 --- TABLE 4b (Toxic, band) ===
Difficulty  Count   STA   Sim   Flu  Joint
      Easy    518 0.917 0.935 0.547  0.771
    Medium    336 0.890 0.939 0.532  0.748
      Hard    296 0.797 0.922 0.535  0.697
 

## Step E3 — Aggregate: Mean+/-Std across seeds + Bootstrap CI

In [16]:
import numpy as np

METRICS = ['STA', 'Sim', 'Flu', 'Joint']

# ========== Mean+-Std across seeds ==========
def print_main_table(prefix, index_col='Phase'):
    dfs = {}
    for seed in SEED_LIST:
        path = os.path.join(RESULTS_DIR, f'{prefix}_seed{seed}.csv')
        if os.path.exists(path):
            dfs[seed] = pd.read_csv(path).set_index(index_col)
    if not dfs:
        print(f'Khong tim thay file {prefix}_seed*.csv'); return
    keys = list(dfs[SEED_LIST[0]].index)
    for k in keys:
        print(f'--- {k} ---')
        for metric in METRICS:
            values, used_seeds = [], []
            for seed in SEED_LIST:
                if seed in dfs and k in dfs[seed].index and metric in dfs[seed].columns:
                    values.append(dfs[seed].loc[k, metric]); used_seeds.append(seed)
            if not values: continue
            seed_str = ' | '.join(f'S{s}={v:.3f}' for s, v in zip(used_seeds, values))
            mean_v = np.mean(values)
            std_v  = np.std(values, ddof=1) if len(values) > 1 else 0.0
            print(f'  {metric:10s}: {seed_str}  ->  Mean+-Std = {mean_v:.4f} +- {std_v:.4f}')
    print()

for prefix, idx_col in [('table3','Phase'),('table3b','Phase'),('table3c','Phase'),
                          ('table4','Difficulty'),('table4b','Difficulty'),('table4c','Difficulty')]:
    sep = '='*80
    print(sep)
    print(f'BANG CHINH --- {prefix.upper()} (FLAT PPO, Mean+-Std across seeds)')
    print(sep)
    print_main_table(prefix, idx_col)

# ========== Bootstrap CI ==========
def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
    lo = np.percentile(boot_means, (1-ci)/2*100)
    hi = np.percentile(boot_means, (1+ci)/2*100)
    return values.mean(), lo, hi

_all = []
for seed in SEED_LIST:
    p = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
    if os.path.exists(p):
        df = pd.read_csv(p)
        df['seed'] = seed
        df['phase'] = 'Flat PPO'
        _all.append(df)

if _all:
    all_pooled = pd.concat(_all, ignore_index=True)
    print(f'Total pooled rows: {len(all_pooled)}')

    def build_ci_table(df, label_filter=None):
        sub = df if label_filter is None else df[df['label']==label_filter]
        rows = []
        for g in sub['phase'].unique():
            gdf = sub[sub['phase']==g]
            row = {'phase': g, 'n_pooled': len(gdf)}
            for col in METRICS:
                if col not in gdf.columns: continue
                mean_v, lo, hi = bootstrap_ci(gdf[col].values)
                row[col] = round(mean_v, 4)
                row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
            rows.append(row)
        return pd.DataFrame(rows)

    for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NONTOXIC', 'nontoxic')]:
        ci_df = build_ci_table(all_pooled, filt)
        ci_df.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower()}.csv'), index=False)
        print(f'\n=== Bootstrap CI --- {label} ===')
        print(ci_df.to_string(index=False))


BANG CHINH --- TABLE3 (FLAT PPO, Mean+-Std across seeds)
--- Baseline (SFT) ---
  STA       : S42=0.867 | S123=0.867 | S777=0.867  ->  Mean+-Std = 0.8674 +- 0.0000
  Sim       : S42=0.972 | S123=0.972 | S777=0.972  ->  Mean+-Std = 0.9718 +- 0.0000
  Flu       : S42=0.548 | S123=0.548 | S777=0.548  ->  Mean+-Std = 0.5484 +- 0.0000
  Joint     : S42=0.759 | S123=0.759 | S777=0.759  ->  Mean+-Std = 0.7588 +- 0.0000
--- Flat PPO ---
  STA       : S42=0.881 | S123=0.895 | S777=0.893  ->  Mean+-Std = 0.8896 +- 0.0074
  Sim       : S42=0.934 | S123=0.918 | S777=0.926  ->  Mean+-Std = 0.9262 +- 0.0080
  Flu       : S42=0.538 | S123=0.533 | S777=0.539  ->  Mean+-Std = 0.5366 +- 0.0032
  Joint     : S42=0.747 | S123=0.745 | S777=0.751  ->  Mean+-Std = 0.7478 +- 0.0029

BANG CHINH --- TABLE3B (FLAT PPO, Mean+-Std across seeds)
--- Baseline (SFT) ---
  STA       : S42=0.866 | S123=0.866 | S777=0.866  ->  Mean+-Std = 0.8663 +- 0.0000
  Sim       : S42=0.971 | S123=0.971 | S777=0.971  ->  Mean+-Std 

## Step E4 — Edit Behaviour + PPL (FULL / Toxic / Non-toxic, Pooled)

In [17]:
# Edit behaviour
def tokenize_simple(text): return str(text).lower().split()

def levenshtein_token(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[n][m]

def compute_edit_behavior(row):
    x, y = str(row['X']).strip(), str(row['Y']).strip()
    xt, yt = tokenize_simple(x), tokenize_simple(y)
    xl = max(len(xt),1); yl = len(yt)
    return pd.Series({'unchanged': x.lower()==y.lower(),
                      'token_edit_rate': levenshtein_token(xt,yt)/xl,
                      'truncated': yl<xl,
                      'clause_loss': ((xl-yl)/xl) > 0.30})

def behaviour_table(df, label='FULL'):
    if label == 'TOXIC':    sub = df[df['label']=='toxic']
    elif label == 'NONTOXIC': sub = df[df['label']=='nontoxic']
    else:                     sub = df
    m = sub.apply(compute_edit_behavior, axis=1)
    return pd.DataFrame([{
        'Phase': 'Flat PPO', 'n': len(sub),
        'Unchanged (%)': round(m['unchanged'].mean()*100, 1),
        'Token changed (%)': round(m['token_edit_rate'].mean()*100, 1),
        'Truncated (%)': round(m['truncated'].mean()*100, 1),
        'Clause loss >30% (%)': round(m['clause_loss'].mean()*100, 1)
    }])

def ppl_table(df, label='FULL'):
    if label == 'TOXIC':    sub = df[df['label']=='toxic']
    elif label == 'NONTOXIC': sub = df[df['label']=='nontoxic']
    else:                     sub = df
    ppls = pd.Series([compute_perplexity(str(y)) for y in sub['Y']])
    pct  = ((ppls>=20)&(ppls<=150)).mean()
    return {'phase':'Flat PPO','mean_PPL':round(ppls.mean(),1),
            'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)}

if _all:
    for label in ['FULL', 'TOXIC', 'NONTOXIC']:
        if label == 'TOXIC':   sub_all = all_pooled[all_pooled['label']=='toxic']
        elif label=='NONTOXIC': sub_all = all_pooled[all_pooled['label']=='nontoxic']
        else:                   sub_all = all_pooled

        beh = behaviour_table(all_pooled, label)
        beh.to_csv(os.path.join(RESULTS_DIR, f'edit_behavior_{label.lower()}.csv'), index=False)
        print(f'\n=== BANG HANH VI --- {label} ===')
        print(beh.to_string(index=False))

        p = ppl_table(all_pooled, label)
        print(f'\n=== BANG PPL --- {label} ===')
        print(f"Flat PPO: mean={p['mean_PPL']:.1f}, median={p['median_PPL']:.1f}, trong[20,150]={p['pct_in_20_150']:.1%}")
        pd.DataFrame([p]).to_csv(os.path.join(RESULTS_DIR, f'ppl_{label.lower()}.csv'), index=False)

    print("\nHOAN TAT tat ca cac bang!")



=== BANG HANH VI --- FULL ===
   Phase    n  Unchanged (%)  Token changed (%)  Truncated (%)  Clause loss >30% (%)
Flat PPO 3936            2.3               44.9           93.7                  69.9

=== BANG PPL --- FULL ===
Flat PPO: mean=128.7, median=76.9, trong[20,150]=75.9%

=== BANG HANH VI --- TOXIC ===
   Phase    n  Unchanged (%)  Token changed (%)  Truncated (%)  Clause loss >30% (%)
Flat PPO 3450            2.1               46.1           95.0                  71.9

=== BANG PPL --- TOXIC ===
Flat PPO: mean=128.9, median=76.0, trong[20,150]=76.2%

=== BANG HANH VI --- NONTOXIC ===
   Phase   n  Unchanged (%)  Token changed (%)  Truncated (%)  Clause loss >30% (%)
Flat PPO 486            4.1               36.7           84.6                  56.0

=== BANG PPL --- NONTOXIC ===
Flat PPO: mean=127.3, median=83.1, trong[20,150]=74.3%

HOAN TAT tat ca cac bang!


##  Rebuild Tables with STA (indep.) Metric

In [18]:
# =====================================================================================
# Independent Style Transfer Accuracy (STA) Evaluation — FLAT PPO
# 1) Load s-nlp/roberta_toxicity_classifier, rescore STA (indep.) on per_sample_*.csv
# 2) Rebuild table3/3b/3c + table4/4b/4c (add STA (indep.) column)
# 3) Rebuild Bootstrap CI (FULL/TOXIC/NON-TOXIC) (add STA (indep.) column)
# =====================================================================================

import os, re, glob, numpy as np, pandas as pd, torch, gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification

METRICS    = ['STA', 'STA (indep.)', 'Sim', 'Flu', 'Joint']

# -------------------------------------------------------------------------------------
# STEP 1 — Independent STA Scoring on CSV files
# -------------------------------------------------------------------------------------
print('=' * 90); print('STEP 1 — Independent STA Scoring (s-nlp/roberta_toxicity_classifier)'); print('=' * 90)
indep_tokenizer = AutoTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')
indep_model = AutoModelForSequenceClassification.from_pretrained('s-nlp/roberta_toxicity_classifier').to(DEVICE)
indep_model.eval()

id2label_indep = indep_model.config.id2label
TOX_IDX_INDEP = next((int(i) for i, l in id2label_indep.items() if str(l).lower() in {'toxic', 'toxicity'}), 1)

@torch.no_grad()
def compute_sta_indep(texts, batch_size=32):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        enc = indep_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
        probs = torch.softmax(indep_model(**enc).logits.float(), dim=-1)[:, TOX_IDX_INDEP]
        out.extend((1.0 - probs).cpu().tolist())
    return out

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_phase_flat.csv'))) \
          + sorted(glob.glob(os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')))

gap_rows = []
for path in csv_files:
    df = pd.read_csv(path)
    if 'STA (indep.)' not in df.columns:
        df['STA (indep.)'] = compute_sta_indep(df['Y'].tolist())
        df.to_csv(path, index=False)

    fname = os.path.basename(path)
    m = re.search(r'seed(\d+)_phase_flat', fname)
    seed, phase = (m.group(1), 'Flat PPO') if m else ('baseline', 'Baseline (SFT)')
    gap_rows.append({'seed': seed, 'phase': phase, 'n': len(df),
                      'STA (orig.)': round(df['STA'].mean(), 4),
                      'STA (indep.)': round(df['STA (indep.)'].mean(), 4),
                      'gap': round(df['STA'].mean() - df['STA (indep.)'].mean(), 4)})

sta_gap_table = pd.DataFrame(gap_rows).sort_values(['phase', 'seed'])
sta_gap_table.to_csv(os.path.join(RESULTS_DIR, 'sta_independent_comparison.csv'), index=False)
print("\n=== Gap Comparison ===")
print(sta_gap_table.to_string(index=False))

del indep_model
gc.collect(); torch.cuda.empty_cache()

# -------------------------------------------------------------------------------------
# STEP 2 — Rebuild table3 / table4
# -------------------------------------------------------------------------------------
print('\n' + '=' * 90); print('STEP 2 — Rebuild table3/table4 (Add STA indep)'); print('=' * 90)

def summarize2(df):
    return {m: float(df[m].mean()) for m in METRICS if m in df.columns}

BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')
baseline_per_sample = pd.read_csv(BASELINE_CSV)
baseline_metrics_full     = summarize2(baseline_per_sample)
baseline_metrics_toxic    = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'toxic'])
baseline_metrics_nontoxic = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'nontoxic'])

for SEED in SEED_LIST:
    fpath = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase_flat.csv')
    if not os.path.exists(fpath): continue
    df = pd.read_csv(fpath)

    t3  = [{'Phase': 'Baseline (SFT)', **baseline_metrics_full},     {'Phase': 'Flat PPO', **summarize2(df)}]
    t3b = [{'Phase': 'Baseline (SFT)', **baseline_metrics_toxic},    {'Phase': 'Flat PPO', **summarize2(df[df['label'] == 'toxic'])}]
    t3c = [{'Phase': 'Baseline (SFT)', **baseline_metrics_nontoxic}, {'Phase': 'Flat PPO', **summarize2(df[df['label'] == 'nontoxic'])}]

    pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
    pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

    t4, t4b, t4c = [], [], []
    for band in ['easy', 'medium', 'hard']:
        bf, bt, bn = df[df['band']==band], df[(df['band']==band)&(df['label']=='toxic')], df[(df['band']==band)&(df['label']=='nontoxic')]
        if len(bf) > 0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize2(bf)})
        if len(bt) > 0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize2(bt)})
        if len(bn) > 0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize2(bn)})

    t4.append({'Difficulty': 'Overall', 'Count': len(df), **summarize2(df)})
    t4b.append({'Difficulty': 'Overall', 'Count': len(df[df['label']=='toxic']), **summarize2(df[df['label']=='toxic'])})
    t4c.append({'Difficulty': 'Overall', 'Count': len(df[df['label']=='nontoxic']), **summarize2(df[df['label']=='nontoxic'])})

    pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

def print_main_table2(prefix, index_col='Phase'):
    dfs = {S: pd.read_csv(os.path.join(RESULTS_DIR, f'{prefix}_seed{S}.csv')).set_index(index_col) for S in SEED_LIST if os.path.exists(os.path.join(RESULTS_DIR, f'{prefix}_seed{S}.csv'))}
    if not dfs: return
    for k in dfs[SEED_LIST[0]].index:
        print(f'--- {k} ---')
        for metric in METRICS:
            vals = [dfs[S].loc[k, metric] for S in SEED_LIST if S in dfs and metric in dfs[S].columns]
            if not vals: continue
            print(f'  {metric:14s}: ' + ' | '.join(f'S{S}={v:.3f}' for S, v in zip([s for s in SEED_LIST if s in dfs], vals)) + f'  ->  Mean±Std = {np.mean(vals):.4f} ± {(np.std(vals, ddof=1) if len(vals)>1 else 0.0):.4f}')
    print()

for prefix, idx_col in [('table3', 'Phase'), ('table3b', 'Phase'), ('table3c', 'Phase'),
                          ('table4', 'Difficulty'), ('table4b', 'Difficulty'), ('table4c', 'Difficulty')]:
    print('=' * 90); print(f'MAIN TABLE — {prefix} (with STA (indep.))'); print('=' * 90)
    print_main_table2(prefix, idx_col)

# -------------------------------------------------------------------------------------
# STEP 3 — Rebuild Bootstrap CI
# -------------------------------------------------------------------------------------
print('\n' + '=' * 90); print('STEP 3 — Rebuild Bootstrap CI (with STA indep)'); print('=' * 90)

def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, (1 - ci) / 2 * 100), np.percentile(boot_means, (1 + ci) / 2 * 100)
    return values.mean(), lo, hi

_all = []
for SEED in SEED_LIST:
    fpath = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase_flat.csv')
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df['seed'], df['phase'] = SEED, 'Flat PPO'
        _all.append(df)
all_per_sample_pooled = pd.concat(_all, ignore_index=True) if _all else None

if all_per_sample_pooled is not None:
    for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NON-TOXIC', 'nontoxic')]:
        sub = all_per_sample_pooled if filt is None else all_per_sample_pooled[all_per_sample_pooled['label'] == filt]
        rows = []
        for g in sub['phase'].unique():
            gdf = sub[sub['phase'] == g]
            row = {'phase': g, 'n_pooled': len(gdf)}
            for col in METRICS:
                if col not in gdf.columns: continue
                mean, lo, hi = bootstrap_ci(gdf[col].values)
                row[col] = round(mean, 4); row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
            rows.append(row)
        ci_table = pd.DataFrame(rows)
        print(f'\n=== Bootstrap CI — {label} (with STA (indep.)) ===')
        print(ci_table.to_string(index=False))
        ci_table.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower().replace("-", "")}.csv'), index=False)


STEP 1 — Independent STA Scoring (s-nlp/roberta_toxicity_classifier)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Gap Comparison ===
    seed          phase    n  STA (orig.)  STA (indep.)     gap
baseline Baseline (SFT) 1312       0.8674        0.9033 -0.0359
     123       Flat PPO 1312       0.8947        0.9203 -0.0256
      42       Flat PPO 1312       0.8811        0.9135 -0.0325
     777       Flat PPO 1312       0.8929        0.9304 -0.0375

STEP 2 — Rebuild table3/table4 (Add STA indep)
MAIN TABLE — table3 (with STA (indep.))
--- Baseline (SFT) ---
  STA           : S42=0.867 | S123=0.867 | S777=0.867  ->  Mean±Std = 0.8674 ± 0.0000
  STA (indep.)  : S42=0.903 | S123=0.903 | S777=0.903  ->  Mean±Std = 0.9033 ± 0.0000
  Sim           : S42=0.972 | S123=0.972 | S777=0.972  ->  Mean±Std = 0.9718 ± 0.0000
  Flu           : S42=0.548 | S123=0.548 | S777=0.548  ->  Mean±Std = 0.5484 ± 0.0000
  Joint         : S42=0.759 | S123=0.759 | S777=0.759  ->  Mean±Std = 0.7588 ± 0.0000
--- Flat PPO ---
  STA           : S42=0.881 | S123=0.895 | S777=0.893  ->  Mean±Std = 0.8896 ± 0.0074
  STA (indep